# Demo 5.2: KV transfer one level up: the broader goal, with the next operation never written down

**Question.** In Demos 5 and 5.1 every sub-goal is spelled out in the prompt, so the goal heads can re-read it
verbatim. What if the prompt names only the **broader goal** (an algorithm) and the next operation has to be
*derived* from it? If the goal heads re-read the goal from the task span here too, swapping their keys / values over
the algorithm's **name** should make the model carry out a different algorithm, with every line of it written by the
model itself.

**Setup.** Two algorithms that share a natural code prefix; the prompt only says `Implement <name> in <language>`.
The prefill is the shared prefix, cut at the indentation token before the first line where the two algorithms
diverge (the **pivot**):

```
p_src: "Implement merge sort in Python."  + if len(arr) <= 1: return arr   ->  mid = len(arr) // 2 ...
p_dst: "Implement quick sort in C++."     + if (arr.size() <= 1) {...}     ->  int pivot = arr[...] ...
```

**Intervention.** The KV transfer of Demo 5, with the transplanted window = the **algorithm name** only
(`span=name`, `\n`-padded to equal length; it works across languages because nothing else is transplanted): from the
pivot on, the goal heads read SRC's keys / values over "merge sort" (values × 4) instead of DST's "quick sort".
Success = the **whole function** the patched C++ run writes implements merge sort.

**Paper.** App. `app:nrgs` (`fig:kv-goal`, `tab:kv-goal`). 12 algorithm pairs × 2 directions × 4 language pairs = 96
instances: causal shift 0.77 (74 / 96), readout shift 0.80.

Needs one 80 GB GPU (Qwen2.5-Coder-32B-Instruct in bf16). The executed outputs are saved in this file.

**The setup** (the paper's figure for this very pair):

![Broader-goal KV transfer: transplanting the goal heads' view of "merge sort" into a C++ quick-sort prompt makes the model write a C++ merge sort (App. I).](figures/05.2_unspecified_op.png)

*Broader-goal KV transfer (`fig:kv-goal`): (i) SRC, merge sort in Python; (ii) DST, quick sort in C++; (iii) DST with
the goal heads reading SRC's k/v over the algorithm name: a merge sort in C++, and the readout at the pivot moves too.*

In [1]:
# repo root = first parent dir that contains src/
import os, sys
_d = os.getcwd()
while _d != '/' and not os.path.isdir(os.path.join(_d, 'src')):
    _d = os.path.dirname(_d)
sys.path.insert(0, _d); os.chdir(_d)
print('repo root:', _d)

# no progress bars: keeps tqdm / HF widget outputs out of the saved notebook
os.environ['TQDM_DISABLE'] = '1'
import functools, tqdm.std, tqdm.notebook, transformers, huggingface_hub
for _c in (tqdm.std.tqdm, tqdm.notebook.tqdm):   # tqdm.notebook ignores TQDM_DISABLE, so force disable=True
    _c.__init__ = functools.partialmethod(_c.__init__, disable=True)
transformers.utils.logging.disable_progress_bar()
huggingface_hub.utils.disable_progress_bars()

repo root: /disk/u/arnab/Codes/Projects/goals


## Config
**Not a script on `clean`.** The reported numbers come from `notebooks/diff/05.4_unspecified.ipynb` on `master`; this
demo re-implements that notebook's run for one pair on top of `src/nrgs` (same prompts, same intervention, same
oracle prompts). Its knobs, kept here: `span=name`, V × 4, no OV boost, greedy 512 tokens, 64-token readouts.

**Head count (a known paper discrepancy).** That run patched and read out through **230 heads**, the 250 goal heads
restricted to layers ≥ 30; the paper says 250. The reported arm below is the 230-head one; we add the full 250-head
set and a random-250 control (neither is in the paper).

**Why this pair.** `merge_quick`, SRC merge sort in Python → DST quick sort in C++, the pair of `fig:kv-goal`. In the
reported sweep it is valid (clean SRC = merge sort, clean DST = quick sort), the patched run implements merge sort,
and the patched readout names merge sort.

In [2]:
CUDA     = '6'
MODEL    = 'Qwen/Qwen2.5-Coder-32B-Instruct'
G250     = 'head_sets/Qwen2.5-Coder-32B-Instruct/goal_topk/goal__lam20_top250.json'
C250     = 'head_sets/Qwen2.5-Coder-32B-Instruct/control/control250__laterhalf.json'
SRC_LANG, DST_LANG = 'python', 'cpp'
V_SCALE    = 4.0       # x the transplanted values read by the patched heads
MAX_NEW    = 512       # greedy tokens per generation (the whole function)
RO_MAX_NEW = 64        # tokens per GoalScope verbalization
ORACLE_MODEL = 'claude-sonnet-4-6'

INSTRUCTION = 'Implement {name} in {lang}. Name the function `func`.'
LANG_NAME = {'python': 'Python', 'cpp': 'C++'}
# one pair of master's PAIRS: SRC = a, DST = b; the prefill (per language) is natural for both algorithms and ends at
# the indentation token before the first diverging line
PAIR = dict(
    a=dict(id='merge_sort', name='merge sort'),
    b=dict(id='quick_sort', name='quick sort'),
    prefill={'python': '<code>\ndef func(arr):\n    if len(arr) <= 1:\n        return arr\n   ',
             'cpp': '<code>\nstd::vector<int> func(std::vector<int> arr) {\n    if (arr.size() <= 1) {\n        return arr;\n    }\n   '},
)

os.environ['CUDA_VISIBLE_DEVICES'] = CUDA

## Load the model

In [3]:
import json, torch
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich.text import Text
from rich.markup import escape

import src.nrgs.functional as FUNC
import src.nrgs.kv_transfer as PP
from src.nrgs.eval import classify_concept

console = Console(width=140, highlight=False)
torch.set_grad_enabled(False)

ctx = FUNC.setup(MODEL, device_map={'': 0})
load = lambda p: [tuple(h) for h in json.load(open(p))]
goal250, ctrl250 = load(G250), load(C250)
goal230 = [(l, h) for l, h in goal250 if l >= 30]          # the reported run's head set
ctx.gsp = None
print(f'goal heads: 250 -> {len(goal230)} in layers >= 30 | control heads: {len(ctrl250)}')

2026-11-03 00:26:41 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Qwen/Qwen2.5-Coder-32B-Instruct
n_heads=40  n_kv_heads=8  group=5  head_dim=128  n_layer=64
goal heads: 250 -> 230 in layers >= 30 | control heads: 250


## 1. The contrastive pair
`FUNC.get_prompt` builds each chat prompt (system prompt + `Implement <name> in <language>.` + the prefill).
`FUNC.token_span` locates the algorithm name in each, and `FUNC.align_kv_spans` pads the shorter name with `\n`
tokens so the two windows line up token for token. That window is all that will be transplanted.

In [4]:
src_alg, dst_alg = PAIR['a'], PAIR['b']
question = lambda alg, lang: INSTRUCTION.format(name=alg['name'], lang=LANG_NAME[lang])
src_prompt = FUNC.get_prompt(question(src_alg, SRC_LANG), PAIR['prefill'][SRC_LANG], ctx=ctx)
dst_prompt = FUNC.get_prompt(question(dst_alg, DST_LANG), PAIR['prefill'][DST_LANG], ctx=ctx)
s0 = FUNC.token_span(src_prompt, src_alg['name'], ctx=ctx)
d0 = FUNC.token_span(dst_prompt, dst_alg['name'], ctx=ctx)
src_tok, dst_tok, src_w, dst_w = FUNC.align_kv_spans(ctx.mt, src_prompt, dst_prompt, s0, d0, ctx=ctx)
g = dict(src_tok=src_tok, dst_tok=dst_tok, src_w=src_w, dst_w=dst_w)   # what PP.intervention_stack needs

def show(tok, win, title, color):
    ids = tok['input_ids'][0].tolist()
    start = max(i for i, t in enumerate(ids) if t == ctx.mt.tokenizer.convert_tokens_to_ids('<|im_start|>') and i < min(win))
    txt = Text()
    for i in range(start, len(ids)):
        s = ctx.mt.tokenizer.decode([ids[i]])
        style = 'bold black on #fde68a' if i == len(ids) - 1 else f'bold {color} on #f1f5f9' if i in win else 'dim'
        txt.append(s, style=style)
    console.rule(title); console.print(txt)

show(src_tok, set(src_w), f'SRC: {src_alg["name"]} ({SRC_LANG}), k/v donor', 'cyan')
show(dst_tok, set(dst_w), f'DST: {dst_alg["name"]} ({DST_LANG}), the run we patch', 'magenta')
print(f'window: {len(dst_w)} tokens ({[ctx.mt.tokenizer.decode([i]) for i in dst_tok["input_ids"][0, dst_w].tolist()]}); '
      f'pivot = {ctx.mt.tokenizer.decode([int(dst_tok["input_ids"][0, -1])])!r}')

─────────────────────────────────────────────────── SRC: merge sort (python), k/v donor ────────────────────────────────────────────────────

<|im_start|>user
Implement merge sort in Python. Name the function `func`.<|im_end|>
<|im_start|>assistant
<code>
def func(arr):
    if len(arr) <= 1:
        return arr
   

───────────────────────────────────────────────── DST: quick sort (cpp), the run we patch ──────────────────────────────────────────────────

<|im_start|>user
Implement quick sort in C++. Name the function `func`.<|im_end|>
<|im_start|>assistant
<code>
std::vector<int> func(std::vector<int> arr) {
    if (arr.size() <= 1) {
        return arr;
    }
   

window: 2 tokens ([' quick', ' sort']); pivot = '   '


## 2. Clean runs
Greedy continuations of both prompts (`FUNC.greedy`) and a GoalScope readout at each pivot (`FUNC.goalscope_readout`,
the 230 heads, additive_norm α 0.1, layers 35–45).

In [5]:
readout = lambda tok, iface=None: FUNC.goalscope_readout('', tok, tok['input_ids'].shape[1] - 1, heads=goal230,
                                                          iface=iface, maxnew=RO_MAX_NEW, ctx=ctx)[0]['patched_gen']
gens = dict(src=FUNC.greedy(src_tok, max_new=MAX_NEW, ctx=ctx), dst=FUNC.greedy(dst_tok, max_new=MAX_NEW, ctx=ctx))
ros = dict(src=readout(src_tok), dst=readout(dst_tok))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


pos=77 '   ' => I'm not executing any code; I'm just preparing to divide the task into smaller parts and merge 
them back together. No action is taken at this line specifically.

pos=91 '   ' => I'm not executing any code directly; I'm selecting a pivot element and partitioning the array 
around it, then recursively sorting the sub-arrays.

## 3. Patched runs
`PP.intervention_stack` is Demo 5's intervention on our window: the patched heads read SRC's k/v over the name
(values × 4) from the pivot on, for the prompt's last query and every generated token. The same stack wraps the
GoalScope readout at the patched pivot.

In [6]:
arms = {'goal 230, V x4 (reported)': goal230, 'goal 250, V x4': goal250, 'control 250, V x4': ctrl250}
for name, heads in arms.items():
    stack = PP.intervention_stack(g, goal_heads=heads, v_scale=V_SCALE, ctx=ctx)
    def iface(mt_, inputs, patches=None, model_kwargs=None, stack=stack):
        with stack():
            return mt_._model(**dict(inputs), **dict(model_kwargs or {}))
    with stack():
        gens[name] = FUNC.greedy(dst_tok, max_new=MAX_NEW, ctx=ctx)
    ros[name] = readout(dst_tok, iface=iface)

pos=91 '   ' => I'm not executing any code directly; I'm here to help you sort and merge arrays. Please provide 
the array you want to sort, and I'll demonstrate the process.

pos=91 '   ' => I'm not executing any code directly; I'm here to help you sort and merge arrays. Please provide 
the array you want to sort, and I'll guide you through the process.

pos=91 '   ' => I'm not executing any code directly; I'm selecting a pivot element and partitioning the array 
around it, then recursively sorting the sub-arrays.

## 4. Judge
The reported run's oracle (`classify_concept`, the same prompts as `05.4_unspecified.ipynb`): for each run it reads
the **whole function** (prefill + continuation up to `</code>`) and says whether it implements SRC's algorithm, DST's,
both (a mix), or neither; the pivot readouts are classified the same way. Claude calls are cached on disk.

In [7]:
ORACLE_TASK = ('Write a function `func` (in Python or C++) that implements a named algorithm. The function starts '
               'with a prefilled prefix; judge the algorithm the function implements')
LABEL = {'src': 'SRC', 'dst': 'DST', 'both': 'both', 'neither': 'neither', 'parse_error': 'parse_error'}
lang_of = lambda k: SRC_LANG if k == 'src' else DST_LANG

def oracle_code(gen, lang):
    fn = PAIR['prefill'][lang].replace('<code>\n', '', 1) + gen.split('</code>')[0]
    r = classify_concept(fn, f"implements {src_alg['name']}", f"implements {dst_alg['name']}", ORACLE_TASK,
                         kind=f'a {LANG_NAME[lang]} function: a prefilled prefix followed by the model\'s continuation. '
                              'Judge which ALGORITHM the function as a whole implements (BOTH = a mix of the two algorithms)',
                         model_name=ORACLE_MODEL)
    return dict(r, verdict=LABEL[r['label']])

def oracle_readout(text):
    r = classify_concept(text, f"implements {src_alg['name']}", f"implements {dst_alg['name']}", ORACLE_TASK,
                         kind='a short natural-language description of what the model is doing at its current '
                              'position in the code (a verbalized readout of its internal state)',
                         model_name=ORACLE_MODEL)
    return dict(r, verdict=LABEL[r['label']])

v_code = {k: oracle_code(gen, lang_of(k)) for k, gen in gens.items()}
v_ro = {k: oracle_readout(ro) for k, ro in ros.items()}

colors = {'src': 'cyan', 'dst': 'magenta'}
for k, gen in gens.items():
    body = gen.split('</code>')[0].rstrip()
    lines = body.splitlines()
    body = '\n'.join(lines[:14] + ([f'... (+{len(lines) - 14} lines)'] if len(lines) > 14 else []))
    title = {'src': f'clean SRC ({SRC_LANG})', 'dst': f'clean DST ({DST_LANG})'}.get(k, f'patched DST: {k}')
    console.print(Panel(Text(body), title=f'[bold {colors.get(k, "white")}]{title}[/]  oracle: [bold]{v_code[k]["verdict"]}[/]',
                        title_align='left', border_style='grey42'))

t = Table(title=f'SRC = {src_alg["name"]} ({SRC_LANG})   |   DST = {dst_alg["name"]} ({DST_LANG})', show_lines=True)
t.add_column('run'); t.add_column('function implements'); t.add_column('GoalScope readout at the pivot', max_width=70)
t.add_column('readout names')
fmt = lambda v: {'SRC': f'[bold cyan]{src_alg["name"]} (SRC)[/]', 'DST': f'[bold magenta]{dst_alg["name"]} (DST)[/]'}.get(v, f'[red]{v}[/]')
for k in gens:
    t.add_row(k if k in arms else f'clean {k.upper()}', fmt(v_code[k]['verdict']), escape(ros[k].strip()), fmt(v_ro[k]['verdict']))
console.print(t)

╭─ clean SRC (python)  oracle: SRC ────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│  mid = len(arr) // 2                                                                                                                     │
│     left_half = func(arr[:mid])                                                                                                          │
│     right_half = func(arr[mid:])                                                                                                         │
│     return merge(left_half, right_half)                                                                                                  │
│                                                                                                                                          │
│ def merge(left, right):                                                                                                                  │
│     sorted_array = []                                                                                                                    │
│     left_index, right_index = 0, 0                                                                                                       │
│     while left_index < len(left) and right_index < len(right):                                                                           │
│         if left[left_index] < right[right_index]:                                                                                        │
│             sorted_array.append(left[left_index])                                                                                        │
│             left_index += 1                                                                                                              │
│         else:                                                                                                                            │
│             sorted_array.append(right[right_index])                                                                                      │
│ ... (+4 lines)                                                                                                                           │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ clean DST (cpp)  oracle: DST ───────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│  int pivot = arr[arr.size() / 2];                                                                                                        │
│     std::vector<int> left, right, equal;                                                                                                 │
│     for (int num : arr) {                                                                                                                │
│         if (num < pivot) {                                                                                                               │
│             left.push_back(num);                                                                                                         │
│         } else if (num > pivot) {                                                                                                        │
│             right.push_back(num);                                                                                                        │
│         } else {                                                                                                                         │
│             equal.push_back(num);                                                                                                        │
│         }                                                                                                                                │
│     }                                                                                                                                    │
│     left = func(left);                                                                                                                   │
│     right = func(right);                                                                                                                 │
│     left.insert(left.end(), equal.begin(), equal.end());                                                                                 │
│ ... (+3 lines)                                                                                                                           │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ patched DST: goal 230, V x4 (reported)  oracle: SRC ────────────────────────────────────────────────────────────────────────────────────╮
│  int mid = arr.size() / 2;                                                                                                               │
│     std::vector<int> left(arr.begin(), arr.begin() + mid);                                                                               │
│     std::vector<int> right(arr.begin() + mid, arr.end());                                                                                │
│     left = func(left);                                                                                                                   │
│     right = func(right);                                                                                                                 │
│     arr.clear();                                                                                                                         │
│     while (!left.empty() || !right.empty()) {                                                                                            │
│         if (!left.empty() && (!right.empty() && left.front() <= right.front())) {                                                        │
│             arr.push_back(left.front());                                                                                                 │
│             left.erase(left.begin());                                                                                                    │
│         } else {                                                                                                                         │
│             arr.push_back(right.front());                                                                                                │
│             right.erase(right.begin());                                                                                                  │
│         }                                                                                                                                │
│ ... (+3 lines)                                                                                                                           │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ patched DST: goal 250, V x4  oracle: SRC ───────────────────────────────────────────────────────────────────────────────────────────────╮
│  int mid = arr.size() / 2;                                                                                                               │
│     std::vector<int> left(arr.begin(), arr.begin() + mid);                                                                               │
│     std::vector<int> right(arr.begin() + mid, arr.end());                                                                                │
│     left = func(left);                                                                                                                   │
│     right = func(right);                                                                                                                 │
│     arr.clear();                                                                                                                         │
│     while (!left.empty() || !right.empty()) {                                                                                            │
│         if (left.empty()) {                                                                                                              │
│             arr.push_back(right.front());                                                                                                │
│             right.erase(right.begin());                                                                                                  │
│         } else if (right.empty()) {                                                                                                      │
│             arr.push_back(left.front());                                                                                                 │
│             left.erase(left.begin());                                                                                                    │
│         } else if (left.front() < right.front()) {                                                                                       │
│ ... (+9 lines)                                                                                                                           │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ patched DST: control 250, V x4  oracle: DST ────────────────────────────────────────────────────────────────────────────────────────────╮
│  int pivot = arr[arr.size() / 2];                                                                                                        │
│     std::vector<int> left, middle, right;                                                                                                │
│     for (int num : arr) {                                                                                                                │
│         if (num < pivot) {                                                                                                               │
│             left.push_back(num);                                                                                                         │
│         } else if (num == pivot) {                                                                                                       │
│             middle.push_back(num);                                                                                                       │
│         } else {                                                                                                                         │
│             right.push_back(num);                                                                                                        │
│         }                                                                                                                                │
│     }                                                                                                                                    │
│     left = func(left);                                                                                                                   │
│     right = func(right);                                                                                                                 │
│     left.insert(left.end(), middle.begin(), middle.end());                                                                               │
│ ... (+3 lines)                                                                                                                           │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                           SRC = merge sort (python)   |   DST = quick sort (cpp)                                           
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ run                       ┃ function implements ┃ GoalScope readout at the pivot                                      ┃ readout names    ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ clean SRC                 │ merge sort (SRC)    │ I'm not executing any code; I'm just preparing to divide the task   │ merge sort (SRC) │
│                           │                     │ into smaller parts and merge them back together. No action is taken │                  │
│                           │                     │ at this line specifically.                                          │                  │
├───────────────────────────┼─────────────────────┼─────────────────────────────────────────────────────────────────────┼──────────────────┤
│ clean DST                 │ quick sort (DST)    │ I'm not executing any code directly; I'm selecting a pivot element  │ quick sort (DST) │
│                           │                     │ and partitioning the array around it, then recursively sorting the  │                  │
│                           │                     │ sub-arrays.                                                         │                  │
├───────────────────────────┼─────────────────────┼─────────────────────────────────────────────────────────────────────┼──────────────────┤
│ goal 230, V x4 (reported) │ merge sort (SRC)    │ I'm not executing any code directly; I'm here to help you sort and  │ neither          │
│                           │                     │ merge arrays. Please provide the array you want to sort, and I'll   │                  │
│                           │                     │ demonstrate the process.                                            │                  │
├───────────────────────────┼─────────────────────┼─────────────────────────────────────────────────────────────────────┼──────────────────┤
│ goal 250, V x4            │ merge sort (SRC)    │ I'm not executing any code directly; I'm here to help you sort and  │ neither          │
│                           │                     │ merge arrays. Please provide the array you want to sort, and I'll   │                  │
│                           │                     │ guide you through the process.                                      │                  │
├───────────────────────────┼─────────────────────┼─────────────────────────────────────────────────────────────────────┼──────────────────┤
│ control 250, V x4         │ quick sort (DST)    │ I'm not executing any code directly; I'm selecting a pivot element  │ quick sort (DST) │
│                           │                     │ and partitioning the array around it, then recursively sorting the  │                  │
│                           │                     │ sub-arrays.                                                         │                  │
└───────────────────────────┴─────────────────────┴─────────────────────────────────────────────────────────────────────┴──────────────────┘

How to read it. Nothing in the destination prompt mentions how to sort: the prefix is shared, and the goal heads only
see a different *name*. Yet the C++ run now splits the array at `mid`, recurses on both halves and merges them: the
goal heads re-read the broader goal from the prompt at the pivot, and the model derives the next operation from
whatever they read. The same transfer through 250 random heads leaves DST on quick sort. The patched **readout** is
weaker than the code: it mentions merging but not the split, and the oracle grades it *neither* (the paper's pooled
readout shift is 0.80, so this pair is one of the misses here; see the cross-check).

## 5. Cross-check against the reported run
The reported run wrote no result cache; its verdicts live in the saved outputs of `master:notebooks/diff/05.4_unspecified.ipynb`
(sweep cell, row `python→cpp · merge_quick · merge_sort → quick_sort`), and panel (iii) of the paper figure shows the
patched code and readout. We compare our 230-head arm with the sweep row, and every arm's code with panel (iii).

In [8]:
REPORTED = dict(clean='SRC / DST', patched='SRC', readout='SRC / DST -> SRC',
                pooled='causal 0.77 (74/96), readout 0.80; py->py 0.79, py->cpp 0.75, cpp->py 0.79, cpp->cpp 0.75')
# panel (iii) of fig:kv-goal, transcribed from the figure (the patched C++ after the prefill)
FIG_PATCHED = '''int mid = arr.size() / 2;
std::vector<int> left(arr.begin(), arr.begin() + mid);
std::vector<int> right(arr.begin() + mid, arr.end());
left = func(left);
right = func(right);
arr.clear();
while (!left.empty() || !right.empty()) {
if (left.empty()) {
arr.push_back(right.front());
right.erase(right.begin());
} else if (right.empty()) {
arr.push_back(left.front());
left.erase(left.begin());
} else if (left.front() < right.front()) {
arr.push_back(left.front());
left.erase(left.begin());
} else {
arr.push_back(right.front());
right.erase(right.begin());
}
}
return arr;
}'''.splitlines()
FIG_READOUT = 'I am here to help you by dividing your request into smaller parts, sorting those parts, and then merging them back together in order.'

arm = 'goal 230, V x4 (reported)'
ours = dict(clean=f"{v_code['src']['verdict']} / {v_code['dst']['verdict']}", patched=v_code[arm]['verdict'],
            readout=f"{v_ro['src']['verdict']} / {v_ro['dst']['verdict']} -> {v_ro[arm]['verdict']}")
t = Table(title='this notebook (230-head arm) vs. the reported sweep row')
t.add_column('quantity'); t.add_column('reported'); t.add_column('ours'); t.add_column('match')
for k in ('clean', 'patched', 'readout'):
    t.add_row(k, escape(REPORTED[k]), escape(ours[k]), '[green]yes[/]' if REPORTED[k] == ours[k] else '[red]no[/]')
console.print(t)

code_lines = lambda gen: [ln.strip() for ln in gen.split('</code>')[0].splitlines() if ln.strip()]
t = Table(title='each patched arm vs. panel (iii) of fig:kv-goal')
t.add_column('arm'); t.add_column('code lines identical to the figure'); t.add_column('first differing line (ours)')
for name in arms:
    ours_l = code_lines(gens[name])
    n = next((i for i, (a, b) in enumerate(zip(ours_l, FIG_PATCHED)) if a != b), min(len(ours_l), len(FIG_PATCHED)))
    same = ours_l == FIG_PATCHED
    t.add_row(name, f'{n} / {len(FIG_PATCHED)}' + ('  [green]all[/]' if same else ''),
              '-' if same else escape(ours_l[n] if n < len(ours_l) else '<end>'))
console.print(t)
print('figure readout:', FIG_READOUT)
print('pooled (96 instances, from the saved sweep):', REPORTED['pooled'])

   this notebook (230-head arm) vs. the reported sweep row    
┏━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ quantity ┃ reported         ┃ ours                 ┃ match ┃
┡━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│ clean    │ SRC / DST        │ SRC / DST            │ yes   │
│ patched  │ SRC              │ SRC                  │ yes   │
│ readout  │ SRC / DST -> SRC │ SRC / DST -> neither │ no    │
└──────────┴──────────────────┴──────────────────────┴───────┘

                                              each patched arm vs. panel (iii) of fig:kv-goal                                               
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ arm                       ┃ code lines identical to the figure ┃ first differing line (ours)                                             ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ goal 230, V x4 (reported) │ 7 / 23                             │ if (!left.empty() && (!right.empty() && left.front() <= right.front())) │
│                           │                                    │ {                                                                       │
│ goal 250, V x4            │ 23 / 23  all                       │ -                                                                       │
│ control 250, V x4         │ 0 / 23                             │ int pivot = arr[arr.size() / 2];                                        │
└───────────────────────────┴────────────────────────────────────┴─────────────────────────────────────────────────────────────────────────┘

figure readout: I am here to help you by dividing your request into smaller parts, sorting those parts, and then merging them back together in order.
pooled (96 instances, from the saved sweep): causal 0.77 (74/96), readout 0.80; py->py 0.79, py->cpp 0.75, cpp->py 0.79, cpp->cpp 0.75


## Where this goes in the paper / how to reproduce at scale
- **Source**: `notebooks/diff/05.4_unspecified.ipynb` on `master` (12 algorithm pairs in `PAIRS`, both directions,
  4 language pairs, the sweep cell + the success-rate cells; cell 19 writes `kv_goal_table.tex`). No script, job list or
  result cache on `clean`; see CLAUDE.md section 4.11.
- **Paper numbers**: causal shift 0.77 (74 / 96), readout shift 0.80; by language pair 0.79 / 0.75 / 0.79 / 0.75
  (py→py, py→cpp, cpp→py, cpp→cpp). The run used **230** heads (G250 in layers ≥ 30), the paper says 250.
- **Figure**: `fig:kv-goal` (`Figures/transfer/unspecified_next-crop.pdf`) is hand-made from this pair. Its patched
  code (panel iii) matches the **250-head** arm line for line (section 5), not the 230-head arm behind the table; its
  patched readout matches neither arm.
- **Related**: Demo 5 (the sub-goal, spelled out in the task list) and Demo 5.1 (one level down, the sub-sub-goal).